<a href="https://colab.research.google.com/github/MaiAlhusseini/FlyRank_Intern_repo/blob/main/Copy_of_w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MaiAlhusseini/FlyRank_Intern_repo/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### Rule

Prioritize a page for refresh when it shows both:
1. signs of performance decline
2. enough time since its last update to make a refresh reasonable.

Signals used:
- trend_pct <= -30% → meaningful recent decline
- days_since_last_update >= 90 days → content is stale enough to review

Scoring:
- +1 point for meaningful decline
- +1 point for stale content

Action:
- score = 2 → refresh
- score = 1 → monitor
- score = 0 → no action

Reason codes:
- DECLINING_AND_STALE
- DECLINING
- STALE
- NO_FLAG

In [ ]:
from google.colab import files

uploaded = files.upload()

import pandas as pd

df = pd.read_csv("content_refresh_anonymized.csv")

Saving content_refresh_anonymized.csv to content_refresh_anonymized (1).csv


In [ ]:
print("Signal 1: days_since_last_update")
print(df["days_since_last_update"].describe())

print("\nSignal 2: trend_pct")
print(df["trend_pct"].describe())

print("\nStaleness buckets:")
stale_bins = [0, 30, 60, 90, 180, float("inf")]
stale_labels = ["0-30", "31-60", "61-90", "91-180", "181+"]
print(
    pd.cut(
        df["days_since_last_update"],
        bins=stale_bins,
        labels=stale_labels,
        include_lowest=True
    ).value_counts().sort_index()
)

print("\nTrend buckets:")
trend_bins = [-float("inf"), -50, -30, -10, 10, 30, float("inf")]
trend_labels = ["<-50%", "-50 to -30%", "-30 to -10%", "-10 to 10%",
                "10 to 30%", ">30%"]

print(
    pd.cut(
        df["trend_pct"],
        bins=trend_bins,
        labels=trend_labels
    ).value_counts().sort_index()
)

Signal 1: days_since_last_update
count    30000.000000
mean        46.098300
std         42.078709
min          1.000000
25%         20.000000
50%         20.000000
75%        104.000000
max        373.000000
Name: days_since_last_update, dtype: float64

Signal 2: trend_pct
count    26612.000000
mean        -4.785969
std        473.861780
min       -100.000000
25%        -62.600000
50%        -33.500000
75%          0.000000
max      44900.000000
Name: trend_pct, dtype: float64

Staleness buckets:
days_since_last_update
0-30      20480
31-60       128
61-90        47
91-180     9171
181+        174
Name: count, dtype: int64

Trend buckets:
trend_pct
<-50%          9646
-50 to -30%    4492
-30 to -10%    4136
-10 to 10%     3013
10 to 30%      1641
>30%           3684
Name: count, dtype: int64


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
df["baseline_score"] = (
    (df["trend_pct"] <= -30).astype(int)
    + (df["days_since_last_update"] >= 90).astype(int)
)

df["reason_code"] = "No_Flag"

df.loc[
    (df["trend_pct"] <= -30) &
    (df["days_since_last_update"] >= 90),
    "reason_code"
] = "Declining_And_Stale"

df.loc[
    (df["trend_pct"] <= -30) &
    (df["days_since_last_update"] < 90),
    "reason_code"
] = "Declining"

df.loc[
    (df["trend_pct"] > -30) &
    (df["days_since_last_update"] >= 90),
    "reason_code"
] = "Stale"


df["action"] = df["baseline_score"].map({
    2: "refresh",
    1: "monitor",
    0: "no_action"
})

ranked = df.sort_values(
    ["baseline_score", "trend_pct", "days_since_last_update"],
    ascending=[False, True, False]
).reset_index(drop=True)

ranked["rank"] = ranked.index + 1


output_cols = [
    "content_id",
    "baseline_score",
    "reason_code",
    "action",
    "trend_pct",
    "trend_direction",
    "days_since_last_update",
    "content_age_days"
]

ranked[output_cols].to_csv(
    "baseline_action_score.csv",
    index=False
)

print(ranked[output_cols].head(20))
print("\nScore distribution:")
print(ranked["baseline_score"].value_counts().sort_index())

              content_id  baseline_score          reason_code   action  \
0   content_f6fdf87348f6               2  Declining_And_Stale  refresh   
1   content_1b4ec72dafd4               2  Declining_And_Stale  refresh   
2   content_7a888d3d99c8               2  Declining_And_Stale  refresh   
3   content_94991fe6268c               2  Declining_And_Stale  refresh   
4   content_ab18b5811c02               2  Declining_And_Stale  refresh   
5   content_84d12054c0c0               2  Declining_And_Stale  refresh   
6   content_ccfb4d0227b1               2  Declining_And_Stale  refresh   
7   content_bbca724138f2               2  Declining_And_Stale  refresh   
8   content_f4b3081037b3               2  Declining_And_Stale  refresh   
9   content_24abafed9707               2  Declining_And_Stale  refresh   
10  content_b694314765e5               2  Declining_And_Stale  refresh   
11  content_277fa742f704               2  Declining_And_Stale  refresh   
12  content_c44f9b88d117              

In [ ]:
import os

os.makedirs("work/outputs", exist_ok=True)

In [ ]:
output_path = "work/outputs/baseline_action_score.csv"

ranked[output_cols].to_csv(output_path, index=False)

print(f"Saved to: {output_path}")

Saved to: work/outputs/baseline_action_score.csv


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
top20_review = ranked[
    [
        "content_id",
        "baseline_score",
        "reason_code",
        "action",
        "trend_pct",
        "trend_direction",
        "days_since_last_update",
        "content_age_days"
    ]
].head(20).copy()

top20_review["confidence"] = "High"

top20_review["what_would_make_it_wrong"] = (
    "Recent performance may be affected by temporary factors, "
    "tracking issues, or missing data."
)

top20_review

,content_id,baseline_score,reason_code,action,trend_pct,trend_direction,days_since_last_update,content_age_days,confidence,what_would_make_it_wrong
0,content_f6fdf87348f6,2,Declining_And_Stale,refresh,-100.0,down,373,373,High,Recent performance may be affected by temporar...
1,content_1b4ec72dafd4,2,Declining_And_Stale,refresh,-100.0,down,372,372,High,Recent performance may be affected by temporar...
2,content_7a888d3d99c8,2,Declining_And_Stale,refresh,-100.0,down,313,313,High,Recent performance may be affected by temporar...
3,content_94991fe6268c,2,Declining_And_Stale,refresh,-100.0,down,313,313,High,Recent performance may be affected by temporar...
4,content_ab18b5811c02,2,Declining_And_Stale,refresh,-100.0,down,305,313,High,Recent performance may be affected by temporar...
5,content_84d12054c0c0,2,Declining_And_Stale,refresh,-100.0,down,304,305,High,Recent performance may be affected by temporar...
6,content_ccfb4d0227b1,2,Declining_And_Stale,refresh,-100.0,down,301,301,High,Recent performance may be affected by temporar...
7,content_bbca724138f2,2,Declining_And_Stale,refresh,-100.0,down,236,236,High,Recent performance may be affected by temporar...
8,content_f4b3081037b3,2,Declining_And_Stale,refresh,-100.0,down,231,237,High,Recent performance may be affected by temporar...
9,content_24abafed9707,2,Declining_And_Stale,refresh,-100.0,down,231,231,High,Recent performance may be affected by temporar...


### Top-20 Review

The top 20 pages were prioritized because they have both a meaningful recent performance decline and enough time since their last update to justify a refresh.

All top-20 pages received a baseline score of 2 and the reason code `DECLINING_AND_STALE`, meaning they satisfy both conditions of the rule.

The recommendations should still be reviewed before action because temporary traffic changes, tracking problems, or missing data could make a page appear to be declining when it is not.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

### Weak Picks + Leakage Check

The top picks are generally consistent with the baseline rule because they show a `trend_pct` of -30% or lower and have not been updated for at least 90 days.

The strongest candidates are therefore pages where both signals are clearly present.

Potentially weak picks would be pages with missing performance data or unusual tracking values, because a missing or unreliable signal could create an artificial decline.

For leakage, the baseline rule uses `trend_pct` and `days_since_last_update`. The review should confirm that these values are based only on information available for the intended evaluation period and do not use future performance or product/action flags.

In [ ]:
# Check the top 20 for weak or suspicious picks

print("Top-20 pages with missing key signals:")
print(
    ranked.head(20)[
        ["content_id", "trend_pct", "days_since_last_update"]
    ].isna().sum()
)

print("\nTop-20 pages:")
print(
    ranked.head(20)[
        [
            "content_id",
            "baseline_score",
            "reason_code",
            "action",
            "trend_pct",
            "days_since_last_update"
        ]
    ]
)

print("\nChecking baseline rule:")
print(
    "All top-20 pages satisfy trend_pct <= -30%:",
    (ranked.head(20)["trend_pct"] <= -30).all()
)

print(
    "All top-20 pages satisfy days_since_last_update >= 90:",
    (ranked.head(20)["days_since_last_update"] >= 90).all()
)

Top-20 pages with missing key signals:
content_id                0
trend_pct                 0
days_since_last_update    0
dtype: int64

Top-20 pages:
              content_id  baseline_score          reason_code   action  \
0   content_f6fdf87348f6               2  Declining_And_Stale  refresh   
1   content_1b4ec72dafd4               2  Declining_And_Stale  refresh   
2   content_7a888d3d99c8               2  Declining_And_Stale  refresh   
3   content_94991fe6268c               2  Declining_And_Stale  refresh   
4   content_ab18b5811c02               2  Declining_And_Stale  refresh   
5   content_84d12054c0c0               2  Declining_And_Stale  refresh   
6   content_ccfb4d0227b1               2  Declining_And_Stale  refresh   
7   content_bbca724138f2               2  Declining_And_Stale  refresh   
8   content_f4b3081037b3               2  Declining_And_Stale  refresh   
9   content_24abafed9707               2  Declining_And_Stale  refresh   
10  content_b694314765e5           

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.